In [4]:
import duckdb

# Start an in-memory DuckDB session
con = duckdb.connect()

# Query the raw CSV directly and display 5 rows
query = """
SELECT * FROM read_csv_auto('../data/raw/olist_orders_dataset.csv') 
LIMIT 5;
"""

# Output as a clean Pandas dataframe
display(con.execute(query).df())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


# Data Profiling (Generic EDA for the raw datasets)

In [2]:
import duckdb
import os
import pandas as pd

# Connect to a fast, temporary in-memory DuckDB instance
con = duckdb.connect()

raw_dir = "../data/raw/"
files = [f for f in os.listdir(raw_dir) if f.endswith('.csv')]

print(f"Found {len(files)} raw CSV datasets. Processing profiles...\n")

for file in files:
    table_name = file.replace('.csv', '')
    file_path = os.path.join(raw_dir, file)
    
    # 1. Total row count
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{file_path}')").fetchone()[0]
    
    # 2. Extract column metadata (names and inferred types)
    columns_df = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{file_path}')").df()
    
    # 3. Dynamically calculate Nulls and Uniqueness for every single column
    metrics = []
    for _, row in columns_df.iterrows():
        col_name = row['column_name']
        col_type = row['column_type']
        
        # SQL to count distinct and null values
        stats = con.execute(f"""
            SELECT 
                COUNT(DISTINCT "{col_name}") as unique_count,
                COUNT(*) - COUNT("{col_name}") as null_count
            FROM read_csv_auto('{file_path}')
        """).fetchone()
        
        metrics.append({
            "Column Name": col_name,
            "Data Type": col_type,
            "Unique Values": stats[0],
            "Null Values": stats[1],
            "Null %": round((stats[1] / total_rows) * 100, 2)
        })
    
    # Print the report for this table
    print("=" * 60)
    print(f"TABLE: {table_name}")
    print(f"Total Rows: {total_rows:,}")
    print("=" * 60)
    display(pd.DataFrame(metrics))
    print("\n" + ("#" * 60) + "\n")

Found 9 raw CSV datasets. Processing profiles...

TABLE: olist_customers_dataset
Total Rows: 99,441


,Column Name,Data Type,Unique Values,Null Values,Null %
0,customer_id,VARCHAR,99441,0,0.0
1,customer_unique_id,VARCHAR,96096,0,0.0
2,customer_zip_code_prefix,VARCHAR,14994,0,0.0
3,customer_city,VARCHAR,4119,0,0.0
4,customer_state,VARCHAR,27,0,0.0



############################################################

TABLE: olist_geolocation_dataset
Total Rows: 1,000,163


,Column Name,Data Type,Unique Values,Null Values,Null %
0,geolocation_zip_code_prefix,VARCHAR,19015,0,0.0
1,geolocation_lat,DOUBLE,717372,0,0.0
2,geolocation_lng,DOUBLE,717615,0,0.0
3,geolocation_city,VARCHAR,8011,0,0.0
4,geolocation_state,VARCHAR,27,0,0.0



############################################################

TABLE: olist_orders_dataset
Total Rows: 99,441


...


############################################################

TABLE: olist_sellers_dataset
Total Rows: 3,095


,Column Name,Data Type,Unique Values,Null Values,Null %
0,seller_id,VARCHAR,3095,0,0.0
1,seller_zip_code_prefix,VARCHAR,2246,0,0.0
2,seller_city,VARCHAR,611,0,0.0
3,seller_state,VARCHAR,23,0,0.0



############################################################

TABLE: product_category_name_translation
Total Rows: 71


,Column Name,Data Type,Unique Values,Null Values,Null %
0,product_category_name,VARCHAR,71,0,0.0
1,product_category_name_english,VARCHAR,71,0,0.0



############################################################

